In [1]:
# IMPORTS ET CONFIGURATION
import asyncio
import os
import sqlite3
import warnings
from typing import List, Optional

from fastapi import FastAPI, HTTPException, Path, Query, status
from fastapi.responses import HTMLResponse, RedirectResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
import uvicorn

# Masquer les avertissements de dépréciation
warnings.filterwarnings("ignore", category=DeprecationWarning)


# Détection dynamique du fichier de base de données
def get_db_path() -> str:
  possible_paths = [
      "data/database/irve_database.db",
      "../data/database/irve_database.db",
      "irve_database.db",
      "irve.db",
  ]
  for path in possible_paths:
    if os.path.exists(path):
      return path
  raise FileNotFoundError("Fichier de base de données IRVE introuvable !")


DB_PATH = get_db_path()
print(f"Base de données chargée depuis : {DB_PATH}")

Base de données chargée depuis : ../data/database/irve_database.db


c:\Users\cgboh\OneDrive\Desktop\irve-bornes-recharge-ml_Charmelle\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [2]:
# MODÈLES PYDANTIC (SCHÉMAS)

class CommuneResponse(BaseModel):
  code_insee: str = Field(
      ..., description="Code INSEE de la commune", examples=["75056"]
  )
  nom_commune: str = Field(
      ..., description="Nom de la commune", examples=["Paris"]
  )
  nbr_stations: Optional[int] = Field(
      None, description="Nombre de stations", examples=[12]
  )


class StationResponse(BaseModel):
  id_station: str = Field(
      ...,
      description="Identifiant unique de la station",
      examples=["FRMGPP92051A"],
  )
  nom_station: str = Field(
      ...,
      description="Nom de la station",
      examples=["Station Paris Central"],
  )
  code_insee: str = Field(
      ..., description="Code INSEE de la commune", examples=["75056"]
  )


class PointDeChargeCreate(BaseModel):
  id_station: str = Field(
      ...,
      description="Identifiant unique de la station rattachée",
      json_schema_extra={"example": "FRMGPP92051A"},
  )
  puissance_nominale: float = Field(
      ...,
      gt=0,
      description="Puissance maximale du point de charge en kW",
      json_schema_extra={"example": 150.0},
  )


class PointDeChargeUpdate(BaseModel):
  puissance_nominale: float = Field(
      ...,
      gt=0,
      description="Nouvelle puissance maximale en kW",
      json_schema_extra={"example": 180.0},
  )


class PointDeChargeResponse(BaseModel):
  id_pdc: str = Field(
      ...,
      description="Identifiant unique du point de charge",
      json_schema_extra={"example": "FRMGPP92051AP1"},
  )
  id_station: str = Field(
      ...,
      description="Identifiant de la station",
      json_schema_extra={"example": "FRMGPP92051A"},
  )
  puissance_nominale: float = Field(
      ...,
      description="Puissance maximale en kW",
      json_schema_extra={"example": 150.0},
  )

class StatsResponse(BaseModel):
    total_communes: int = Field(
        ..., description="Nombre total de communes", examples=[11703]
    )
    total_stations: int = Field(
        ..., description="Nombre total de stations", examples=[48581]
    )
    total_points_de_charge: int = Field(
        ..., description="Nombre total de points de charge", examples=[166189]
    )

In [3]:
# DÉFINITION DE L'API FASTAPI
app = FastAPI(
    title="API IRVE - Infrastructure de Recharge",
    description="""
    API REST pour gérer le réseau de bornes et points de charge pour véhicules électriques (IRVE).

    ## Fonctionnalités principales :
    * **Communes** : Consultation des communes et nombre de stations.
    * **Stations** : Recherche et fiches détaillées des stations.
    * **Points de charge** : Gestion CRUD complète du matériel.
    * **Statistiques** : Vue d'ensemble du réseau.
    """,
    version="1.0.0",
)


# --- GÉNÉRAL ---
@app.get("/", tags=["Général"], summary="Redirection vers Swagger")
def read_root():
  """Redirige automatiquement la racine '/' vers Swagger '/docs'."""
  return RedirectResponse(url="/docs")


# --- STATISTIQUES ---
@app.get(
    "/stats",
    response_model=StatsResponse,  # Affiche le schéma détaillé au lieu de 'any'
    tags=["Statistiques"],
    summary="Obtenir les statistiques globales",
)
def get_stats():
  """Récupère le nombre total de communes, de stations et de points de charge
  référencés."""
  conn = sqlite3.connect(DB_PATH)
  cursor = conn.cursor()
  cursor.execute("SELECT COUNT(*) FROM COMMUNES")
  nb_communes = cursor.fetchone()[0]
  cursor.execute("SELECT COUNT(*) FROM STATIONS")
  nb_stations = cursor.fetchone()[0]
  cursor.execute("SELECT COUNT(*) FROM POINTS_DE_CHARGE")
  nb_pdc = cursor.fetchone()[0]
  conn.close()

  return {
      "total_communes": nb_communes,
      "total_stations": nb_stations,
      "total_points_de_charge": nb_pdc,
  }

# --- COMMUNES ---
@app.get(
    "/communes",
    response_model=List[CommuneResponse],
    tags=["Communes"],
    summary="Lister les communes",
)
def get_communes(limit: int = Query(10, ge=1, le=1000)):
  conn = sqlite3.connect(DB_PATH)
  cursor = conn.cursor()
  try:
    cursor.execute(
        "SELECT code_insee, nom_commune, nbr_stations FROM COMMUNES LIMIT ?",
        (limit,),
    )
    rows = cursor.fetchall()
    result = [
        {"code_insee": r[0], "nom_commune": r[1], "nbr_stations": r[2]}
        for r in rows
    ]
  except sqlite3.OperationalError:
    cursor.execute(
        "SELECT code_insee, nom_commune FROM COMMUNES LIMIT ?", (limit,)
    )
    rows = cursor.fetchall()
    result = [
        {"code_insee": r[0], "nom_commune": r[1], "nbr_stations": None}
        for r in rows
    ]
  conn.close()
  return result


@app.get(
    "/communes/{code_insee}",
    response_model=CommuneResponse,
    tags=["Communes"],
    summary="Détail d'une commune",
)
def get_commune_detail(
    code_insee: str = Path(
        ..., description="Code INSEE de la commune", examples=["92051"]
    )
):
  conn = sqlite3.connect(DB_PATH)
  cursor = conn.cursor()
  try:
    cursor.execute(
        "SELECT code_insee, nom_commune, nbr_stations FROM COMMUNES WHERE"
        " code_insee = ?",
        (code_insee,),
    )
    row = cursor.fetchone()
    result = (
        {"code_insee": row[0], "nom_commune": row[1], "nbr_stations": row[2]}
        if row
        else None
    )
  except sqlite3.OperationalError:
    cursor.execute(
        "SELECT code_insee, nom_commune FROM COMMUNES WHERE code_insee = ?",
        (code_insee,),
    )
    row = cursor.fetchone()
    result = (
        {"code_insee": row[0], "nom_commune": row[1], "nbr_stations": None}
        if row
        else None
    )
  conn.close()

  if not result:
    raise HTTPException(
        status_code=status.HTTP_404_NOT_FOUND, detail="Commune non trouvée"
    )

  return result


# --- STATIONS ---
@app.get(
    "/stations",
    response_model=List[StationResponse],
    tags=["Stations"],
    summary="Lister les stations",
)
def get_stations(
    code_insee: Optional[str] = None, limit: int = Query(10, ge=1, le=1000)
):
  conn = sqlite3.connect(DB_PATH)
  cursor = conn.cursor()
  if code_insee:
    cursor.execute(
        "SELECT id_station, nom_station, code_insee FROM STATIONS WHERE"
        " code_insee = ? LIMIT ?",
        (code_insee, limit),
    )
  else:
    cursor.execute(
        "SELECT id_station, nom_station, code_insee FROM STATIONS LIMIT ?",
        (limit,),
    )
  rows = cursor.fetchall()
  conn.close()

  return [
      {"id_station": r[0], "nom_station": r[1], "code_insee": r[2]}
      for r in rows
  ]


@app.get(
    "/stations/{id_station}",
    response_model=StationResponse,
    tags=["Stations"],
    summary="Détail d'une station",
)
def get_station_detail(
    id_station: str = Path(
        ...,
        description="Identifiant unique de la station",
        examples=["FRMGPP92051A"],
    )
):
  conn = sqlite3.connect(DB_PATH)
  cursor = conn.cursor()
  cursor.execute(
      "SELECT id_station, nom_station, code_insee FROM STATIONS WHERE"
      " id_station = ?",
      (id_station,),
  )
  row = cursor.fetchone()
  conn.close()

  if not row:
    raise HTTPException(
        status_code=status.HTTP_404_NOT_FOUND, detail="Station introuvable"
    )

  return {"id_station": row[0], "nom_station": row[1], "code_insee": row[2]}


# --- POINTS DE CHARGE ---
@app.get(
    "/points-de-charge",
    response_model=List[PointDeChargeResponse],
    tags=["Points de Charge"],
    summary="Lister les points de charge",
)
def get_points_de_charge(
    puissance_min: Optional[float] = Query(
        None, description="Puissance minimale en kW"
    ),
    limit: int = Query(10, ge=1, le=1000),
):
  conn = sqlite3.connect(DB_PATH)
  cursor = conn.cursor()
  if puissance_min is not None:
    cursor.execute(
        "SELECT id_pdc, id_station, puissance_nominale FROM POINTS_DE_CHARGE"
        " WHERE puissance_nominale >= ? LIMIT ?",
        (puissance_min, limit),
    )
  else:
    cursor.execute(
        "SELECT id_pdc, id_station, puissance_nominale FROM POINTS_DE_CHARGE"
        " LIMIT ?",
        (limit,),
    )
  rows = cursor.fetchall()
  conn.close()

  return [
      {"id_pdc": r[0], "id_station": r[1], "puissance_nominale": r[2]}
      for r in rows
  ]


@app.get(
    "/points-de-charge/{id_pdc}",
    response_model=PointDeChargeResponse,
    tags=["Points de Charge"],
    summary="Obtenir le détail d'un point de charge",
)
def get_point_de_charge(
    id_pdc: str = Path(
        ...,
        description="Identifiant du point de charge",
        examples=["FRMGPP92051AP1"],
    )
):
  conn = sqlite3.connect(DB_PATH)
  cursor = conn.cursor()
  cursor.execute(
      "SELECT id_pdc, id_station, puissance_nominale FROM POINTS_DE_CHARGE"
      " WHERE id_pdc = ?",
      (id_pdc,),
  )
  pdc = cursor.fetchone()
  conn.close()

  if not pdc:
    raise HTTPException(
        status_code=status.HTTP_404_NOT_FOUND,
        detail="Point de charge introuvable",
    )

  return {"id_pdc": pdc[0], "id_station": pdc[1], "puissance_nominale": pdc[2]}


@app.post(
    "/points-de-charge",
    response_model=PointDeChargeResponse,
    status_code=status.HTTP_201_CREATED,
    tags=["Points de Charge"],
    summary="Créer un nouveau point de charge",
)
def create_point_de_charge(pdc: PointDeChargeCreate):
  conn = sqlite3.connect(DB_PATH)
  cursor = conn.cursor()

  # 1. Vérifier la station
  cursor.execute(
      "SELECT id_station FROM STATIONS WHERE id_station = ?", (pdc.id_station,)
  )
  if not cursor.fetchone():
    conn.close()
    raise HTTPException(
        status_code=status.HTTP_404_NOT_FOUND,
        detail="Station référencée introuvable",
    )

  # 2. Chercher le premier ID libre
  cursor.execute(
      "SELECT COUNT(*) FROM POINTS_DE_CHARGE WHERE id_station = ?",
      (pdc.id_station,),
  )
  index = cursor.fetchone()[0] + 1

  while True:
    candidate_id = f"{pdc.id_station}P{index}"
    cursor.execute(
        "SELECT 1 FROM POINTS_DE_CHARGE WHERE id_pdc = ?", (candidate_id,)
    )
    if not cursor.fetchone():
      new_id_pdc = candidate_id
      break
    index += 1

  # 3. Insertion
  cursor.execute(
      "INSERT INTO POINTS_DE_CHARGE (id_pdc, id_station, puissance_nominale)"
      " VALUES (?, ?, ?)",
      (new_id_pdc, pdc.id_station, pdc.puissance_nominale),
  )

  conn.commit()
  conn.close()

  return {
      "id_pdc": new_id_pdc,
      "id_station": pdc.id_station,
      "puissance_nominale": pdc.puissance_nominale,
  }


@app.put(
    "/points-de-charge/{id_pdc}",
    response_model=PointDeChargeResponse,
    tags=["Points de Charge"],
    summary="Mettre à jour la puissance d'un point de charge",
)
def update_point_de_charge(
    id_pdc: str, update_data: PointDeChargeUpdate
):  # Correctif : string au lieu de int
  conn = sqlite3.connect(DB_PATH)
  cursor = conn.cursor()

  cursor.execute(
      "UPDATE POINTS_DE_CHARGE SET puissance_nominale = ? WHERE id_pdc = ?",
      (update_data.puissance_nominale, id_pdc),
  )

  if cursor.rowcount == 0:
    conn.close()
    raise HTTPException(
        status_code=status.HTTP_404_NOT_FOUND,
        detail="Point de charge introuvable",
    )

  cursor.execute(
      "SELECT id_pdc, id_station, puissance_nominale FROM POINTS_DE_CHARGE"
      " WHERE id_pdc = ?",
      (id_pdc,),
  )
  pdc = cursor.fetchone()
  conn.commit()
  conn.close()

  return {"id_pdc": pdc[0], "id_station": pdc[1], "puissance_nominale": pdc[2]}


@app.delete(
    "/points-de-charge/{id_pdc}",
    tags=["Points de Charge"],
    summary="Supprimer un point de charge",
)
def delete_point_de_charge(id_pdc: str):
  conn = sqlite3.connect(DB_PATH)
  cursor = conn.cursor()

  cursor.execute("DELETE FROM POINTS_DE_CHARGE WHERE id_pdc = ?", (id_pdc,))

  if cursor.rowcount == 0:
    conn.close()
    raise HTTPException(
        status_code=status.HTTP_404_NOT_FOUND,
        detail="Point de charge introuvable",
    )

  conn.commit()
  conn.close()
  return {"message": f"Point de charge {id_pdc} supprimé avec succès"}

In [4]:

# TESTS AUTOMATISÉS
client = TestClient(app)

print("--- 1. Test GET / (Redirection Swagger) ---")
res = client.get("/", follow_redirects=False)
print(f"Status Code : {res.status_code}")
print(f"Redirection : {res.headers.get('location')}")

print("\n--- 2. Test GET /stats ---")
print(client.get("/stats").json())

print("\n--- 3. Test GET /communes ---")
print(client.get("/communes?limit=2").json())

print("\n--- 4. Test GET /stations ---")
stations = client.get("/stations?limit=2").json()
print(stations)

if stations:
  sample_id = stations[0]["id_station"]
  print(f"\n--- 5. Test GET /stations/{sample_id} ---")
  print(client.get(f"/stations/{sample_id}").json())

  print(f"\n--- 6. Test POST /points-de-charge ---")
  post_res = client.post(
      "/points-de-charge",
      json={"id_station": sample_id, "puissance_nominale": 150.0},
  )
  print(f"Status POST : {post_res.status_code}")
  print(f"Response POST : {post_res.json()}")

  if post_res.status_code == 201:
    new_pdc_id = post_res.json()["id_pdc"]

    print(f"\n--- 7. Test PUT /points-de-charge/{new_pdc_id} ---")
    put_res = client.put(
        f"/points-de-charge/{new_pdc_id}", json={"puissance_nominale": 180.0}
    )
    print(f"Status PUT : {put_res.status_code}")
    print(f"Response PUT : {put_res.json()}")

    print(f"\n--- 8. Test DELETE /points-de-charge/{new_pdc_id} ---")
    del_res = client.delete(f"/points-de-charge/{new_pdc_id}")
    print(f"Status DELETE : {del_res.status_code}")
    print(f"Response DELETE : {del_res.json()}")

--- 1. Test GET / (Redirection Swagger) ---
Status Code : 307
Redirection : /docs

--- 2. Test GET /stats ---
{'total_communes': 11703, 'total_stations': 48581, 'total_points_de_charge': 166189}

--- 3. Test GET /communes ---
[{'code_insee': '92051', 'nom_commune': 'Neuilly-sur-Seine', 'nbr_stations': None}, {'code_insee': '31069', 'nom_commune': 'Blagnac', 'nbr_stations': None}]

--- 4. Test GET /stations ---
[{'id_station': 'FRMGPP92051A', 'nom_station': 'Metropolis - ePremium - Neuilly-sur-Seine - Bretteville', 'code_insee': '92051'}, {'id_station': 'FRMGPP92051E', 'nom_station': 'Metropolis - ePremium - Neuilly-sur-Seine - Achille Peretti', 'code_insee': '92051'}]

--- 5. Test GET /stations/FRMGPP92051A ---
{'id_station': 'FRMGPP92051A', 'nom_station': 'Metropolis - ePremium - Neuilly-sur-Seine - Bretteville', 'code_insee': '92051'}

--- 6. Test POST /points-de-charge ---
Status POST : 201
Response POST : {'id_pdc': 'FRMGPP92051AP9', 'id_station': 'FRMGPP92051A', 'puissance_nominal

In [ ]:
# LANCEMENT DU SERVEUR
config = uvicorn.Config(app, host="127.0.0.1", port=8003)
server = uvicorn.Server(config)
asyncio.create_task(server.serve())

print("🚀 Serveur Uvicorn lancé !")
print("🔗 Swagger Docs : http://127.0.0.1:8003/docs")

🚀 Serveur Uvicorn lancé !
🔗 Swagger Docs : http://127.0.0.1:8003/docs


INFO:     Started server process [22988]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8003 (Press CTRL+C to quit)


INFO:     127.0.0.1:55523 - "GET / HTTP/1.1" 307 Temporary Redirect
INFO:     127.0.0.1:55523 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:55523 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:62266 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:62266 - "GET /openapi.json HTTP/1.1" 200 OK
